# <center>Welcome to 21cmFirstCLASS Notebook on Non Gaussianity!</center>
### <center>By S.Libanore, K. Finish, J. Bernal and J. Flitter </center>

#### In this notebook you will learn how to...
- Use non-Gaussian initial conditions

## General definitions

In [ ]:
# Import the packages required for this tutorial

import numpy as np 
import matplotlib as mpl
import matplotlib.pyplot as plt

import py21cmfast as p21c # To run 21cmFirstCLASS (21cmFAST)
from py21cmfast import plotting # For plotting global signals, coeval boxes, lightcone boxes and power spectra
import py21cmfast.power_spectrum as ps # Calculate power spectrum from the lightcone
from py21cmfast.inputs import global_params # Useful in this tutorial to plot the initial conditions for the simulation
from scipy.interpolate import interp1d
from functools import partial

from scipy.special import erfc
from classy import Class

-----------------------------------------------------
SarahLibanore: developing f_nl, C files updated with IC and fcoll in Lidz approximation on 02/28/2025
-----------------------------------------------------


It is VERY recommended for 21cmFirstCLASS users to have the 'latex' package installed in the same python environment where 21cmFirstCLASS is installed. <br>
If you have latex installed then you may run the next cell, otherwise do not run it as it will raise errors and no plots in this tutorial will be shown!

In [2]:
plt.rcParams.update({"text.usetex": True, "font.family": "Times new roman"}) # Use latex fonts

Define the color palette (this combination is good for colorblindness).

In [3]:
colors =  ['#377eb8', '#ff7f00', '#4daf4a',
           '#f781bf', '#a65628', '#984ea3',
           '#999999', '#e41a1c', '#dede00']

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=colors) # Set the color palette as default

# PART 1:  Check the modeling functions

First of all, we test the functions that should be implemented in the main code to model the non Gaussian contribution. To do so, we need to initialize a CLASS run and obtain the matter power spectrum and transfer function.

In [ ]:
# Cosmological parameters in LCDM

cosmo_params_fNL0 = {"hlittle": 0.6736, # hubble parameter
                "OMb": 0.0493, # baryon density
                "OMm": 0.3153, # matter (CDM+baryon) density
                "A_s": 2.1e-9, # amplitude of the primordial fluctuations
                "POWER_INDEX": 0.9649, # spectral index of the primordial spectrum
                "tau_reio": 0.0544, # optical depth to reionization
                "F_NL":0., # to test matching with old case
                }


In [5]:
CLASS_params = {}
CLASS_params['h'] = cosmo_params_fNL0['hlittle']
CLASS_params['Omega_cdm'] = cosmo_params_fNL0['OMm'] - cosmo_params_fNL0['OMb']
CLASS_params['Omega_b'] = cosmo_params_fNL0['OMb']
CLASS_params['A_s'] = cosmo_params_fNL0['A_s']
CLASS_params['n_s'] = cosmo_params_fNL0['POWER_INDEX']
CLASS_params['output'] = 'tCl,pCl,lCl,mTk,vTk,mPk'
CLASS_params['lensing'] = 'yes'
CLASS_params['z_pk'] = 1087.
CLASS_params['l_max_scalars'] = 3000
# We need to run CLASS for very large wavenumbers. This is required for computing sigma(M) and the HMF
CLASS_params['P_k_max_1/Mpc'] = 1200.

# Run CLASS!
CLASS_OUTPUT = Class()
CLASS_OUTPUT.set(CLASS_params)
CLASS_OUTPUT.compute()

Transfer_0 = CLASS_OUTPUT.get_transfer(z=0.)
k_CLASS = Transfer_0['k (h/Mpc)'][:]*cosmo_params_fNL0['hlittle'] # 1/Mpc
delta_m_0_CLASS = Transfer_0['d_tot'][:]
k_output = pow(10.,np.array(global_params.LOG_K_ARR_FOR_TRANSFERS)) # 1/Mpc
delta_m_0 = p21c.generate_ICs.Interpolate_transfer(delta_m_0_CLASS,k_CLASS,k_output)

We now define three point functions needed to model the non Gaussian corrections to $f_{\rm coll}$.

We define the matrix of the $\langle\delta_n\delta_m^2\rangle$ where $n,m$ are associated with two different masses. In the $f_{\rm coll}$ computation, we always have $M_n<M_m$, hence we define the matrix such that on the upper triangular part we have $\langle\delta_n\delta_m^2\rangle$, while the lower triangular is $\langle\delta_n^2\delta_m\rangle$ and the diagonal $\langle\delta_{n,m}^3\rangle$.

In [ ]:
# this is F_n_local in Eq 27 in my overleaf
def FM(kv,MassVector,cosmo_params,global_params,kcutoff,fnl):

    rho_crit = 2.7754e11 * cosmo_params['hlittle']**2
    rhoM = rho_crit* cosmo_params['OMm']

    Mm = MassVector[None,None,:]
    Rm = (3.*Mm/(4.*np.pi*rhoM))**(1/3.)

    kall = pow(10.,np.array(global_params.LOG_K_ARR_FOR_TRANSFERS)) # 1/Mpc
    k = np.asarray(kall[[kall > kcutoff][0]])
    mu = np.linspace(-0.995, 0.995, 128) # cos theta

    k_2 = k[:, None, None]
    mu_val = mu[None,:,None]

    P_k1 =  9/25. * (2*np.pi**2/kv**3) * cosmo_params['A_s'] * (kv/0.05)**(cosmo_params['POWER_INDEX'] - 1.) # 9/25 is already in the transfer function from CLASS
    P_k2 =  9/25. * (2*np.pi**2/k_2**3) * cosmo_params['A_s'] * (k_2/0.05)**(cosmo_params['POWER_INDEX'] - 1.) # 9/25 is already in the transfer function from CLASS

    x2 = k_2*Rm
    Wm_k2 = 3.0*(np.sin(x2) - x2*np.cos(x2))/(x2)**3  # dimension k1, k2, mu, Mn, Mm
    dWm_k2 = (3 / np.power(x2, 2) * np.sin(x2) - 9 / np.power(x2, 4) * (np.sin(x2) - x2 * np.cos(x2))) * (x2) / (3 * Mm)

    k_CLASS = Transfer_0['k (h/Mpc)'][:]*cosmo_params['hlittle'] # 1/Mpc
    delta_m_0_CLASS = - 5/3. * Transfer_0['d_tot'][:]
    Tm_k1 = interp1d(k_CLASS, delta_m_0_CLASS, kind='cubic', bounds_error=False,fill_value=0.)(kv)
    Tm_k2 = interp1d(k_CLASS, delta_m_0_CLASS, kind='cubic', bounds_error=False,fill_value=0.)(k_2)

    k_12 = np.sqrt(pow(kv,2)+pow(k_2,2) + 2*kv*k_2*mu_val)
     
    Tm_k12 = interp1d(k_CLASS, delta_m_0_CLASS, kind='cubic', bounds_error=False,fill_value=0.)(k_12)

    x12 = k_12 * (3.*MassVector[None,None,:]/(4.*np.pi*rhoM))**(1/3.)
    sin_x12 = np.sin(x12)
    cos_x12 = np.cos(x12)
    Wm_k12 = 3.0*(sin_x12 - x12*cos_x12)/(x12)**3

    dWm_k12 =  (3 / np.power(x12, 2) * np.sin(x12) - 9 / np.power(x12, 4) * (np.sin(x12) - x12 * np.cos(x12))) * (x12) / (3 * Mm)

    P_k12 =  9/25. * (2*np.pi**2/k_12**3) * cosmo_params['A_s'] * (k_12/0.05)**(cosmo_params['POWER_INDEX'] - 1.)

    integrand = k_2**2 * Wm_k2 * Tm_k2 * Wm_k12 * Tm_k12 * (P_k1 * P_k2 + P_k2 * P_k12 + P_k1 * P_k12)
    
    integral_dk2 = np.trapz(integrand, k, axis = 0)
    Fm = np.trapz(integral_dk2, mu, axis = 0)

    Fm *= 2. * fnl / (8*np.pi**4.) * Tm_k1

    integrand_dn2 = k_2**2 * Tm_k2 * Tm_k12 * (P_k1 * P_k2 + P_k2 * P_k12 + P_k1 * P_k12) * (dWm_k2 * Wm_k12 + Wm_k2 * dWm_k12)
    
    integral_dk2_dn2 = np.trapz(integrand_dn2, k, axis = 0)
    dFm_dn2 = np.trapz(integral_dk2_dn2, mu, axis = 0)
    
    dFm_dn2 *= 2. * fnl / (8*np.pi**4.) * Tm_k1

    return [Fm, dFm_dn2]

# this function returns the matrix of the three point function, based on Eq 7 in the overleaf
# and the matrix for the d<>/dM_min (which is built in the same way)
# its elements are <\delta_m^3> on the diagonal, <\delta_m^2\delta_n> on the upper triangular part, <\delta_n^2\delta_m> on the lower triangular part
# read more about the matrix in the overleaf, page 7 
def tpf(MassVector,cosmo_params,global_params,kcutoff,fnl):

    kall = pow(10.,np.array(global_params.LOG_K_ARR_FOR_TRANSFERS)) # 1/Mpc
    k = np.asarray(kall[[kall > kcutoff][0]])

    rho_crit = 2.7754e11 * cosmo_params['hlittle']**2
    rhoM = rho_crit* cosmo_params['OMm']    
    
    k_1 = k[:,None]

    FM_function = lambda kv: FM(kv,MassVector=MassVector,cosmo_params=cosmo_params,global_params=global_params,kcutoff=kcutoff,fnl=fnl)
    temp = np.apply_along_axis(FM_function, axis=-1, arr=k_1)
    Fm = temp[:,0,:][:,None,:]
    dFm_dn2 = temp[:,1,:][:,None,:]

    k_1 = k[:,None,None]
    Mn = MassVector[None,:,None]
    Rn = (3.*Mn/(4.*np.pi*rhoM))**(1/3.)
    x1 = k_1*Rn
    Wn_k1 = 3.0*(np.sin(x1) - x1*np.cos(x1))/(x1)**3 
    dWn_k1 = (3 / np.power(x1, 2) * np.sin(x1) - 9 / np.power(x1, 4) * (np.sin(x1) - x1 * np.cos(x1))) * (x1) / (3 * Mn)
    integrand = k_1 ** 2 * Wn_k1 * Fm

    ddd = np.trapz(integrand,k,axis=0)

    integrand_dnm2 = k_1 ** 2 * dWn_k1 * Fm
    integrand_dmn2 = k_1 ** 2 * Wn_k1 * dFm_dn2
    integrand_dn3 = k_1 ** 2 * (dWn_k1 * Fm + Wn_k1 * dFm_dn2)

    der_dnm2 = np.trapz(integrand_dnm2,k,axis=0)
    der_dmn2 = np.trapz(integrand_dmn2,k,axis=0)
    der_dn3 = np.trapz(integrand_dn3,k,axis=0)

    der_ddd = np.zeros((len(MassVector),len(MassVector)))
    # Fill below the diagonal (i > j) with n2m
    der_ddd[np.tril_indices(len(MassVector), -1)] = der_dmn2[np.tril_indices(len(MassVector), -1)]

    # Fill above the diagonal (i < j) with nm2
    der_ddd[np.triu_indices(len(MassVector), 1)] = der_dnm2[np.triu_indices(len(MassVector), 1)]

    # Fill the diagonal with nnn
    np.fill_diagonal(der_ddd, np.diagonal(der_dn3))

    return ddd, der_ddd

In [ ]:
# this is the mass array 
log10_M_array = np.linspace(4.,20.,300)
rho_crit = 2.7754e11 * cosmo_params_fNL0['hlittle']**2
rhoM = rho_crit* cosmo_params_fNL0['OMm']

In [ ]:
# compute the three point functions and their derivatives wrt M_min
# inputs are the mass vector, the parameters we defined at the beginning
# a cutoff scale (keep the default value) and the value of fnl you want to test

fnl = 300.

tpf, der = tpf(MassVector = 10**log10_M_array,cosmo_params=cosmo_params_fNL0,global_params=global_params,kcutoff=0.01,fnl=fnl)

In [ ]:
def fcoll_DALOSIO(M1_id,M2_id,tpf,z):

    M1 = 10**log10_M_array[M1_id]
    M2 = 10**log10_M_array[M2_id]
    
    Rn = (3.*M1/(4.*np.pi*rhoM))**(1/3.)
    Rm = (3.*M2/(4.*np.pi*rhoM))**(1/3.)

    sigma1 = CLASS_OUTPUT.sigma(Rn,0.)
    sigma2 = CLASS_OUTPUT.sigma(Rm,0.)

    growthf = CLASS_OUTPUT.scale_independent_growth_factor(z)
    delta1 = 1.68
    sigma1 = sigma1*sigma1
    sigma2 = sigma2*sigma2
    delta2 = np.sqrt(sigma2) * growthf

    deltagrowth_diff = ( delta1 - delta2 )/growthf
    if (sigma1 < sigma2):
        return 0.,0.
    elif (sigma1 > sigma2):
        sigma_diff = sigma1 - sigma2
    elif (sigma1==sigma2):
        sigma_diff = 1.e-6
        
    fcoll_dMmin_EPS = erfc((deltagrowth_diff)/np.sqrt( 2.*sigma_diff ) )

    delta_n3 = tpf[M1_id,M1_id]
    delta_m3 = tpf[M2_id,M2_id] 
    delta_m2delta_n = tpf[M1_id,M2_id]
    delta_mdelta_n2 = tpf[M2_id,M1_id]

    A = (delta_n3 - delta_m3 + 3.*delta_m2delta_n - 3.*delta_mdelta_n2)
    B = (delta_m3 + delta_mdelta_n2 - 2*delta_m2delta_n)
    Cval = (delta_m2delta_n - delta_m3)

    dfcoll_G_dSn = deltagrowth_diff / np.sqrt(2*np.pi) / pow(sigma_diff,3/2.) * np.exp(-pow(deltagrowth_diff,2)/2./sigma_diff); 

    val = ( pow(delta1 / growthf, 2) - delta1 * delta2 / pow(growthf,2)) / sigma2 ; 
    cothD = (np.exp(val) + np.exp(-val)) / (np.exp(val) - np.exp(-val)); 

    one_dSn = A/3. * (
         deltagrowth_diff / sigma_diff - 1./deltagrowth_diff )* dfcoll_G_dSn

    two_dSn = B/sigma2 * ( delta1 / growthf - deltagrowth_diff * cothD ) * dfcoll_G_dSn 

    three_dSn = Cval*sigma_diff/(pow(sigma2,2) * deltagrowth_diff ) * (pow(delta2 / growthf, 2.) - sigma2 - 2. * (pow(delta1 / growthf,2)-delta1*delta2/pow(growthf,2)) * (cothD -1.)) * dfcoll_G_dSn  

    if (delta1 <= delta2 or pow(delta1/growthf,2) < sigma2 or sigma2 == 0.):
        fcoll_dMmin_NG = 0.
    else:
        fcoll_dMmin_NG = (one_dSn + two_dSn + three_dSn)
    
    return fcoll_dMmin_EPS , fcoll_dMmin_NG 


At this point, we compute $f_{\rm coll}$ and $df_{\rm coll}/dM_{\rm min}$ following both D'Alosio and Lidz. We want to understand if the Lidz approximation is good, and if and where it breaks.

The inputs in all the functions are:
- M1_id, M2_id = indexes for the mass array (e.g., M1_id = 0 means that you want to use the first value in the mass array)
- tpf = matrix of the three point function 
- der = matrix of the derivative of the three point function
- z = redshift

In [13]:
def fcoll(M1_id,M2_id,tpf,z):

    M1 = 10**log10_M_array[M1_id]
    M2 = 10**log10_M_array[M2_id]
    sigma1 = CLASS_OUTPUT.sigma((3.*M1/(4.*np.pi*rhoM))**(1/3.),0.)
    sigma2 = CLASS_OUTPUT.sigma((3.*M2/(4.*np.pi*rhoM))**(1/3.),0.)

    growthf = CLASS_OUTPUT.scale_independent_growth_factor(z)
    delta1 = 1.68
    sigma1 = sigma1*sigma1
    sigma2 = sigma2*sigma2
    delta2 = np.sqrt(sigma2) * growthf

    deltagrowth_diff = ( delta1 - delta2 )/growthf
    if (sigma1 < sigma2):
        return 0.,0.
    elif (sigma1 > sigma2):
        sigma_diff = sigma1 - sigma2
    elif (sigma1==sigma2):
        sigma_diff = 1.e-6
        
    fcoll_dMmin_EPS = erfc((deltagrowth_diff)/np.sqrt( 2.*sigma_diff ) )

    delta_n3 = tpf[M1_id,M1_id]
    delta_m3 = tpf[M2_id,M2_id] 
    delta_m2delta_n = tpf[M1_id,M2_id]
    delta_mdelta_n2 = tpf[M2_id,M1_id]

    A = (delta_n3 - delta_m3 + 3.*delta_m2delta_n - 3.*delta_mdelta_n2)
    B = (delta_m3 + delta_mdelta_n2 - 2*delta_m2delta_n)

    if (delta1 <= delta2 or pow(delta1/growthf,2) < sigma2 or sigma2 == 0.):
        fcoll_dMmin_NG = 0.
    else:
        fcoll_dMmin_NG = np.exp(-pow(deltagrowth_diff,2)/2./sigma_diff) * (A / 3./np.sqrt(2*np.pi)/sigma_diff**(3/2.)*(deltagrowth_diff**2/sigma_diff - 1.)+ B*delta2/growthf/sigma2*deltagrowth_diff/np.sqrt(2*np.pi)/sigma_diff**(3/2.))
        

    return fcoll_dMmin_EPS , fcoll_dMmin_NG 


In C code we need to implement the derivative of $f_{\rm coll}$. Here we implement it in the full D'Alosio expression and in the Lidz approximation.

In [ ]:
# try to implement this!
def dfcoll_dAlosio():

    return

In [16]:
def dfcoll(M1_id,M2_id,tpf,der,z):

    M1 = 10**log10_M_array[M1_id]
    M2 = 10**log10_M_array[M2_id]
    sigma1 = CLASS_OUTPUT.sigma((3.*M1/(4.*np.pi*rhoM))**(1/3.),0.)
    sigma2 = CLASS_OUTPUT.sigma((3.*M2/(4.*np.pi*rhoM))**(1/3.),0.)

    dsigma_2_dlog10_M = (CLASS_OUTPUT.sigma((3.*(M1*10.**0.1)/(4.*np.pi*rhoM))**(1/3.),0.) - sigma1)/0.1
    dsigmadm =2.*sigma1/(np.log(10.)*M1)*dsigma_2_dlog10_M

    growthf = CLASS_OUTPUT.scale_independent_growth_factor(z)
    delta1 = 1.68
    sigma1 = sigma1*sigma1
    sigma2 = sigma2*sigma2
    delta2 = np.sqrt(sigma2) * growthf
 
    deltagrowth_diff = ( delta1 - delta2 )/growthf
    if (sigma1 < sigma2):
        return 0.,0.
    elif (sigma1 > sigma2):
        sigma_diff = sigma1 - sigma2
    elif (sigma1==sigma2):
        sigma_diff = 1.e-6
        
    dfcoll_dMmin_EPS = (-(deltagrowth_diff)*dsigmadm *( np.exp( - pow( deltagrowth_diff, 2 )/( 2.*sigma_diff ) ) )/(pow(sigma_diff, 1.5)))/np.sqrt(np.pi*2)

    delta_n3 = tpf[M1_id,M1_id]
    delta_m3 = tpf[M2_id,M2_id] 
    delta_m2delta_n = tpf[M1_id,M2_id]
    delta_mdelta_n2 = tpf[M2_id,M1_id]

    ddelta_n3_dMmin = der[M1_id,M1_id]
    ddelta_m2delta_n_dMin = der[M1_id,M2_id]
    ddelta_mdelta_n2_dMin = der[M2_id,M1_id]

    A = (delta_n3 - delta_m3 + 3.*delta_m2delta_n - 3.*delta_mdelta_n2)
    B = (delta_m3 + delta_mdelta_n2 - 2*delta_m2delta_n)

    dA_dMmin = (ddelta_n3_dMmin + 3.*ddelta_m2delta_n_dMin - 3.*ddelta_mdelta_n2_dMin)
    dB_dMmin = (ddelta_mdelta_n2_dMin - 2.*ddelta_m2delta_n_dMin) 

    if (delta1 <= delta2 or pow(delta1/growthf,2) < sigma2 or sigma2 == 0.):
        dfcoll_dMmin_NG = 0.
    else:
        dfdS = (deltagrowth_diff/np.sqrt(2*np.pi)/pow(sigma_diff,3/2.)) * np.exp(-pow(deltagrowth_diff,2)/2./sigma_diff);

        ddfdSdM = dfdS * dsigmadm / 2. / sigma_diff * (pow(deltagrowth_diff,2.)/sigma_diff - 3.);

        term = delta2 / growthf / sigma2;

        one = A / 3. * (deltagrowth_diff / sigma_diff - 1. / deltagrowth_diff) + B * term;

        two = dA_dMmin / 3. * (deltagrowth_diff / sigma_diff - 1. / deltagrowth_diff) - A / 3. * dsigmadm * (deltagrowth_diff / pow(sigma_diff,2.)) + dB_dMmin * term ;

        dfcoll_dMmin_NG = - (ddfdSdM * one + dfdS * two);

    return dfcoll_dMmin_EPS , dfcoll_dMmin_NG 


Now use the following functions to produce the plots we discussed. Good luck :)